# 修了課題DEMO③ 仮想通貨

©2026. For information, contact Deloitte Tohmatsu Group.

[JDLAが策定しているバージョン](https://www.jdla.org/certificate/engineer/)に合わせるために、以下のセルの実行をお願いします．

（#コメントアウト されているものは必要ありません）

また実行完了後に「ランタイムの再起動」をして下さい．

（以下のセルの実行は、最初にしていただければ、以降必要ありません．）

In [ ]:
%%capture
!pip uninstall matplotlib -y
!pip install matplotlib==3.7.1

# !pip uninstall opencv-python -y
# !pip install opencv-python==4.7.0.72

# !pip uninstall torch -y
# !pip install torch==2.0.1

# !pip uninstall torchvision -y
# !pip install torchvision==0.15.2

##はじめに
この修了課題では、仮想通貨の情報を利用して、

未来の仮想通貨の価格予想を行うものです。

今回はDEMOとして、RNN(再帰型ニューラルネットワーク)を用いて学習をさせてみましょう。

##作成までの流れ
大まかな流れとして
1. データのダウンロードと成形

   データダウンロードしてそのまま活用するのは困難です。
   モデルが学習できるよう、カテゴリ変数に置き換えたり、
   欠損値を補完したりなど、成形する必要があります。
2. モデルの構築

   学習を行うモデルのアルゴリズムを理解して、コードを作成します。

3. 学習と結果

   学習を行い結果を確認してみます。
   精度が目標まで達したら、提出用のデータを作成します。

#1.データのダウンロードと成形

## データのダウンロード

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# 学習データのダウンロード
!wget 'https://drive.google.com/uc?export=download&id=1kUfPb8qikA8rdQ26iVUxpod2Qjw3ct_O' -O crypto_train.csv

# テストデータのダウンロード
!wget 'https://drive.google.com/uc?export=download&id=1VhzCcjNSDxGRG86Za653zHHpjVCdSPD3' -O crypto_test_x.csv

In [ ]:
# 学習データの確認
train = pd.read_csv('crypto_train.csv', index_col=0)
train

## データの成形

今回使用するRNNのために必要な処理を施していきます。

In [ ]:
# 正規化処理
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaler = scaler.fit(train.values)

# 正規化する
s_train = scaler.transform(train)
# 元の値に戻す場合
inv_train = scaler.inverse_transform(s_train)

In [ ]:
# データの形状確認
print(s_train.shape)

In [ ]:
# シーケンスデータの教師ありデータを作成する
def make_seq(data, seq_len=6):
    X = data[:, 0:seq_len]
    X = np.expand_dims(X, axis=2)
    Y = data[:, seq_len:]
    return X, Y

X, Y = make_seq(s_train)

def train_val_split(X, Y, val_rate):
  rate = int(X.shape[0]*(1-val_rate))
  train_X, train_Y, val_X, val_Y = X[:rate], Y[:rate], X[rate:], Y[rate:]
  return train_X, train_Y, val_X, val_Y

train_X, train_Y, val_X, val_Y = train_val_split(X, Y, 0.1)

In [ ]:
# データの形状確認
print('train_X:', train_X.shape)
print('train_Y:', train_Y.shape)

# 2.モデルの構築

## 層の設定

In [ ]:
# RNN層の設定
"""
RNN
・再帰ニューラルネットワーク
・コンテキストの単語の並び方を逃していることを対策するための層。
・コンテキストの長さによらず、そのコンテキストの情報を記憶するメカニズムをもつ。
・ループする経路（閉じた経路）をもつのが特徴で、そのことで過去の情報を記憶しながら、最新のデータへと更新できる。
"""

class RNN:
    def __init__(self, Wx, Wh, b):
        #引数でわたされたパラメタをメンバ変数paramsにリストとして設定。
        self.params = [Wx, Wh, b]
        #各パラメータに対応する形で勾配gradsを初期化し、格納
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]
        #逆伝播の計算時に使用する中間データcacheをNoneで初期化
        self.cache = None

    def forward(self, x, h_prev):
        """
        x: (N, D) 現在時刻の入力
        h_prev: (N, H) ひとつ前の隠れ状態
        return: h_next (N, H) 次の隠れ状態
        """
        Wx, Wh, b = self.params

        # RNN基本式: h_next = tanh(x Wx + h_prev Wh + b)
        t = np.dot(h_prev, Wh) + np.dot(x, Wx) + b
        h_next = np.tanh(t)

        # 逆伝播で使うので保持
        self.cache = (x, h_prev, h_next)
        return h_next

    def backward(self, dh_next):
        """
        dh_next: (N, H) 損失Lに対する「次の隠れ状態 h_next」の勾配 dL/dh_next
        return:
          dx      : (N, D) ひとつ前の層への勾配
          dh_prev : (N, H) ひとつ前の時刻の隠れ状態への勾配
        勾配は self.grads にも格納する。
        """
        Wx, Wh, b = self.params
        x, h_prev, h_next = self.cache

        # tanhの微分: d/dt tanh(t) = 1 - tanh(t)^2
        dt = dh_next * (1 - h_next ** 2)

        # バイアス勾配: 時系列方向に和をとる（バッチ方向Nでsum）
        db = np.sum(dt, axis=0)

        # Wh勾配: h_prev^T dt
        dWh = np.dot(h_prev.T, dt)

        # dh_prev: dt Wh^T
        dh_prev = np.dot(dt, Wh.T)

        # Wx勾配: x^T dt
        dWx = np.dot(x.T, dt)

        # dx: dt Wx^T
        dx = np.dot(dt, Wx.T)

        # 勾配を保存（既存配列にブロードキャスト代入してるのはin-place更新）
        self.grads[0][...] = dWx
        self.grads[1][...] = dWh
        self.grads[2][...] = db

        return dx, dh_prev

In [ ]:
# TimeRNN層の設定
# T個(Tは任意の数)のRNNレイヤから構成される層
class TimeRNN:
    def __init__(self, Wx, Wh, b, stateful=False):#statefulはTimeRNN層の隠れ状態を維持するかしないかを意味する。長い時系列データを維持するときはTrueにする必要がある。
        # RNNセルが使う重みを共有するので、TimeRNNも同じparamsをまとめ保持
        self.params = [Wx, Wh, b]
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]
        self.layers = None  # 各時刻tごとにRNNセルを1個ずつ持っておく
        self.h, self.dh = None, None  # hは直前ステップの隠れ状態
        self.stateful = stateful

    def forward(self, xs):
        """
        xs: (N, T, D)
          N: バッチサイズ
          T: 時系列長
          D: 入力次元
        return:
          hs: (N, T, H) 各時刻の隠れ状態を全部並べたもの
        """
        Wx, Wh, b = self.params
        N, T, D = xs.shape
        D, H = Wx.shape

        self.layers = []
        hs = np.empty((N, T, H), dtype='f')#出力用の容器を用意する。

        # stateful=False か、もしくはまだhを持ってない場合はリセット
        if not self.stateful or self.h is None:
            self.h = np.zeros((N, H), dtype='f')


        # 時系列方向にRNNセルを順に適用していく
        for t in range(T):
            layer = RNN(*self.params)                       # 同じ重みで新しいRNNセルを生成
            self.h = layer.forward(xs[:, t, :], self.h)     # 1ステップ分forward
            hs[:, t, :] = self.h                            # 出力を保存
            self.layers.append(layer)                       # 逆伝播用に保持

        return hs

    def backward(self, dhs):
        """
        dhs: (N, T, H)
          各時刻の隠れ状態 h(t) に対する損失勾配 dL/dh(t)
        return:
          dxs: (N, T, D) 入力列xsに対する勾配
        また self.grads にパラメータの合計勾配を格納し、self.dh に「次バッチへ渡すべき h の勾配」を保持。
        """
        Wx, Wh, b = self.params
        N, T, H = dhs.shape
        D, H = Wx.shape

        dxs = np.empty((N, T, D), dtype='f')  # 各時刻のdxを格納する入れ物
        dh = 0                                # 次の時刻(将来)から伝わってくる勾配を蓄積
        grads = [0, 0, 0]                     # Wx, Wh, b の勾配を積算するための一時変数

        # 時刻T-1→0へ逆順にバックプロパゲーション（BPTT）
        for t in reversed(range(T)):
            layer = self.layers[t]
            # 将来からの勾配dhと、今回のdhs[:, t, :]を足してbackward
            dx, dh = layer.backward(dhs[:, t, :] + dh)#合算した勾配
            dxs[:, t, :] = dx

            # 各RNNセルの勾配を足しこんで合算
            for i, grad in enumerate(layer.grads):
                grads[i] += grad

        # 合算した勾配をメンバにコピーしておく
        for i, grad in enumerate(grads):
            self.grads[i][...] = grad

        # 最初の時刻に対する「前の隠れ状態h_prev」側の勾配を記憶
        self.dh = dh

        return dxs

    def set_state(self, h):
        """
        外から隠れ状態hを手動でセットする。（stateful RNNで続きをやりたいときに使う）
        """
        self.h = h

    def reset_state(self):
        """
        隠れ状態hを捨てる。stateful=Trueで学習していても明示的にリセットできる。
        """
        self.h = None

In [ ]:
# Affine層の設定
class Affine:
    """
    通常の全結合層。(N, D) の入力を (N, O) の出力へ線形変換する。
    RNNの最終時刻の隠れ状態から回帰値を出力するのに使う。
    """
    def __init__(self, W, b):
        self.params = [W, b]                 # W: (D, O), b: (O,)
        self.grads = [np.zeros_like(W), np.zeros_like(b)]
        self.x = None                        # forwardで使った入力を保存

    def forward(self, x):
        """
        x: (N, D)
        return: (N, O)
        """
        W, b = self.params
        self.x = x
        return np.dot(x, W) + b

    def backward(self, dout):
        """
        dout: (N, O) = dL/d(out)
        return: dx (N, D)
        さらに self.grads に dW, db を格納
        """
        W, b = self.params

        db = np.sum(dout, axis=0)        # バイアス勾配
        dW = np.dot(self.x.T, dout)      # 重み勾配
        dx = np.dot(dout, W.T)           # 入力側の勾配

        self.grads[0][...] = dW
        self.grads[1][...] = db

        return dx

In [ ]:
# MeanSquaredError損失層
class MeanSquaredError:
    """
    平均二乗誤差(MSE)損失層。
    回帰問題なのでクロスエントロピーではなくMSEを使う。
    L = (1 / (2N)) * Σ (y - t)^2
    """
    def __init__(self):
        self.params, self.grads = [], []  # 学習パラメータはなし
        self.cache = None

    def forward(self, y, t):
        """
        y: (N, O) モデルの予測値
        t: (N, O) 正解値
        return: スカラー損失
        """
        N = y.shape[0]
        loss = np.sum((y - t) ** 2) / (2 * N)
        self.cache = (y, t, N)
        return loss

    def backward(self, dout=1):
        """
        dout: 上流からのスカラー勾配(基本1)
        return: dx (N, O) = dL/dy
        """
        y, t, N = self.cache
        dx = dout * (y - t) / N
        return dx

## 最適化手法の設定

In [ ]:
#  最適化手法の設定
#  確率的勾配降下法（Stochastic Gradient Descent）
class SGD:
    """
    params[i] -= lr * grads[i]
    を繰り返してパラメータを更新する。
    """
    def __init__(self, lr=0.01):
        self.lr = lr

    def update(self, params, grads):
        # 渡された params と grads のリストを同じインデックスで対応させて更新
        for i in range(len(params)):
            params[i] -= self.lr * grads[i]

## モデルの設定

In [ ]:
# モデルSimpleRnnの設定

class SimpleRnn:
    def __init__(self, input_size, hidden_size, output_size):
        D, H, O = input_size, hidden_size, output_size
        rn = np.random.randn  # 正規分布乱数ショートカット

        # 重みの初期化
        # RNN用: 入力→隠れ, 隠れ→隠れ, バイアス
        rnn_Wx = (rn(D, H) / np.sqrt(H)).astype('f')
        rnn_Wh = (rn(H, H) / np.sqrt(H)).astype('f')
        rnn_b = np.zeros(H).astype('f')

        # 出力用Affine: 隠れ→出力 (H×O) とバイアス (O,)
        affine_W = (rn(H, O) / np.sqrt(H)).astype('f')
        affine_b = np.zeros(O).astype('f')

        #レイヤの生成
        self.rnn_layer = TimeRNN(rnn_Wx, rnn_Wh, rnn_b, stateful=False)
        self.affine_layer = Affine(affine_W, affine_b)
        # MSE損失 (回帰版)
        self.loss_layer = MeanSquaredError()

        self.layers = [self.rnn_layer, self.affine_layer]

        # すべてのパラメータと勾配を1つのリストにまとめる
        self.params, self.grads = [], []
        for layer in self.layers:
            self.params += layer.params
            self.grads += layer.grads

    def predict(self, xs):
        """
        xs: (N, T, D) 入力系列
        return: (N, O) 予測値
        """
        hs = self.rnn_layer.forward(xs)
        # 最終時刻の隠れ状態のみ使って回帰値を出力する
        h_last = hs[:, -1, :]
        y = self.affine_layer.forward(h_last)
        return y

    def forward(self, xs, ts):
        """
        xs: (N, T, D) 入力系列
        ts: (N, O)    正解値
        return: 損失スカラー
        """
        y = self.predict(xs)
        loss = self.loss_layer.forward(y, ts)
        return loss

    def backward(self, dout=1):
        """
        MeanSquaredError から逆向きに戻っていって
        各層の勾配を self.grads に反映する。
        """
        dy = self.loss_layer.backward(dout)
        dh_last = self.affine_layer.backward(dy)

        # Affineに渡したのは最終時刻の隠れ状態だけなので、
        # TimeRNNに渡す勾配は (N, T, H) の最終時刻にだけdh_lastを入れる
        N, H = dh_last.shape
        T = len(self.rnn_layer.layers)
        dhs = np.zeros((N, T, H), dtype='f')
        dhs[:, -1, :] = dh_last

        self.rnn_layer.backward(dhs)
        return None

#3.精度確認から提出まで

##学習

In [ ]:
# 学習のハイパーパラメータ
input_size = 1       # 入力データのひとつあたりの次元
hidden_size = 3     # 隠れ状態の次元
output_size = 1      # 出力データのひとつあたりの次元

epochs = 100        # 学習エポック数
learning_rate = 0.1 # 学習率

# データの設定
X = train_X.astype('f')
Y = train_Y.astype('f')

# モデルとオプティマイザの用意
model = SimpleRnn(input_size, hidden_size, output_size)
optimizer = SGD(lr=learning_rate)

# 学習ループ
loss_list = []
for epoch in range(epochs):
    # forward/backward/パラメータ更新
    loss = model.forward(X, Y)  # 損失計算
    model.backward()            # 勾配計算
    optimizer.update(model.params, model.grads)  # SGDでパラメータ更新

    loss_list.append(float(loss))
    print('epoch:', epoch+1, 'Loss:', loss)

In [ ]:
# 検証データで精度を確認する
X = val_X.astype('f')
Y = val_Y.astype('f')

# モデルで予測
pred = model.predict(X)

# スケーラーを使用して、標準化されたデータを元のスケールに戻す
scaler = scaler.fit(train.values.reshape(-1, 1))
Y = scaler.inverse_transform(Y)
pred = scaler.inverse_transform(pred)

# sklearnのmean_squared_error関数を使用してRMSEを計算
from sklearn.metrics import mean_squared_error
import math

# 平均二乗誤差の平方根（RMSE）を計算
rmse = math.sqrt(mean_squared_error(Y, pred))
print('検証RMSE:', rmse)

In [ ]:
plt.plot(Y[:])
plt.plot(pred[:])

今回の検証RMSEの値を確認すると、このモデルの仕様では目標の精度には達していないことが分かります。

他のライブラリをインポートして、より精度が高くなるようなモデルを構築してみると

精度が向上するかもしれません。

## テストデータに対して予測値を出力する
今回学習したRNNが

テスト用のデータを予測して、結果をcsvファイルとして出力するまでを

掲載してみました。

このcsvファイルを「修了課題提出用サイト」にアップロードすると結果を確認することができます。

In [ ]:
# テストデータの確認
test_x = pd.read_csv('crypto_test_x.csv', index_col=0)
test_x

In [ ]:
# テストデータも正規化する
scaler = scaler.fit(test_x)
s_test_x = scaler.transform(test_x)

# データ形式をモデルに対して適正化する
s_test_x = np.expand_dims(s_test_x, axis=2).astype('f')

# モデルで予測
pred = model.predict(s_test_x)

# もとに戻した値
scaler = scaler.fit(test_x.values.reshape(-1, 1))
inv_pred = scaler.inverse_transform(pred)

In [ ]:
# 予測した値を出力する
pred = pd.DataFrame(inv_pred, columns=['Sun'])
pred.to_csv('crypto_pred.csv')
pred